# MF — Diagnostica del fold-in v2

Il riferimento principale è **B, k=10**. I film senza giudizio danno zero guadagno osservato e non sono classificati come dislike. Le curve a k diversi hanno target B diversi. Il test già ispezionato rimane esplorativo. Ogni report indica modalità e sottoinsieme; le repliche sono mediate per utente prima del bootstrap.

Le celle sono intenzionalmente disattivate: attivare soltanto la fase desiderata. Nessun risultato precedente è sovrascritto. Vedere `docs/experiments/recommendation-coldstart-v2.md`.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p/'configs/recommendation_coldstart_v2.json').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Aprire il notebook dalla repository o da notebooks/')
if str(ROOT/'src') not in sys.path:
    sys.path.insert(0, str(ROOT/'src'))
from movie_rag.recommendation_benchmark import BenchmarkConfig, Protocol
CONFIG = BenchmarkConfig.load(ROOT)
PROTOCOL_ID = CONFIG.values['protocol_id']
print('Protocollo:', PROTOCOL_ID, '| configurazione:', CONFIG.fingerprint[:12])


Diagnostica sul validation set con i parametri MF già congelati da full_validation. Rifit esplicito sui soli rating train; nessun accesso al ground truth per modificare i parametri.

In [ ]:
import numpy as np
from movie_rag.recommendation_experiments import load_vectors, load_selection
from movie_rag.recommendation_models import ExplicitMF
RUN_DIAGNOSTIC = False
if RUN_DIAGNOSTIC:
    protocol = Protocol.load(CONFIG)
    _, embedding_hash = load_vectors(protocol)
    chosen = load_selection(protocol, 'baselines', {'embedding': embedding_hash})['winners']['mf_explicit_foldin']['params']
    mf = ExplicitMF(protocol.ids, rank=chosen['rank'], epochs=chosen['epochs'], reg=chosen['reg'],
                    lr=CONFIG.values['mf']['learning_rate'], seed=CONFIG.values['seed']).fit(protocol.train)
    rows = []
    for (sid, k), obs in protocol.observed.items():
        if not sid.startswith('validation_'):
            continue
        bias, factors = mf.fold_in(obs, chosen['fold_reg'])
        rows.append({'scenario_id':sid, 'k':k, 'user_bias':bias,
                     'factor_norm':float(np.linalg.norm(factors)),
                     'personal_score_sd':float(np.std(mf.Q @ factors)),
                     'item_bias_sd':float(np.std(mf.item_bias)),
                     'n_unknown_observed':int(sum(mf.item_counts[mf.index[int(mid)]] == 0 for mid in obs.movieId))})
    display(pd.DataFrame(rows))
else:
    print('Rifit diagnostico disattivato')
